# Fine-tune Mistral-7B with QLoRA for Review Summarization

**Requirements:** Google Colab with T4 GPU (free tier). Optimized to fit in 16GB.

**Checkpoints save to Google Drive** — if Colab disconnects, just reconnect and re-run. Training resumes from last checkpoint.

**Upload:** `summary_training_pairs.jsonl` from `data/processed/`

In [1]:
# Cell 1: Mount Google Drive (checkpoints survive disconnects)
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/mistral-qlora-reviews"
ADAPTER_DIR = "/content/drive/MyDrive/mistral-qlora-adapter"
print(f"Checkpoints will save to: {SAVE_DIR}")

Mounted at /content/drive
Checkpoints will save to: /content/drive/MyDrive/mistral-qlora-reviews


In [2]:
# Cell 2: Install deps and upload data
!pip install -q transformers datasets peft bitsandbytes accelerate trl

from google.colab import files

# Only upload if not already on Drive
import os
if not os.path.exists('/content/drive/MyDrive/summary_training_pairs.jsonl'):
    uploaded = files.upload()  # Upload summary_training_pairs.jsonl
    !cp summary_training_pairs.jsonl /content/drive/MyDrive/
    print('Saved to Google Drive')
else:
    !cp /content/drive/MyDrive/summary_training_pairs.jsonl .
    print('Loaded from Google Drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.6 MB/s eta 0:00:00


Saving summary_training_pairs.jsonl to summary_training_pairs.jsonl
Saved to Google Drive


In [3]:
# Cell 3: Load and format data
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"GPU: {torch.cuda.get_device_name(0)}")

pairs = []
with open('summary_training_pairs.jsonl') as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} training pairs")

def format_prompt(pair):
    return (
        f"<s>[INST] You are an expert product analyst. "
        f"Given the following customer reviews, write a structured executive summary.\n\n"
        f"{pair['input'][:2000]}\n[/INST]\n"
        f"{pair['summary']}</s>"
    )

formatted = [{'text': format_prompt(p)} for p in pairs]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(split['train'])}, Val: {len(split['test'])}")

GPU: Tesla T4
Loaded 400 training pairs
Train: 360, Val: 40


In [4]:
# Cell 4: Load Mistral-7B in 4-bit
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"Model loaded: {MODEL_NAME}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.2


In [5]:
# Cell 5: Apply LoRA (memory-optimized)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470


In [6]:
# Cell 6: Train (saves checkpoints to Google Drive)
import os

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    processing_class=tokenizer,
    args=training_args,
)

# Resume from checkpoint if exists (survives disconnects)
checkpoint = None
if os.path.exists(SAVE_DIR):
    checkpoints = [
        d for d in os.listdir(SAVE_DIR)
        if d.startswith('checkpoint')
    ]
    if checkpoints:
        checkpoint = os.path.join(
            SAVE_DIR, sorted(checkpoints)[-1]
        )
        print(f"Resuming from {checkpoint}")

print("Starting training...")
trainer.train(resume_from_checkpoint=checkpoint)
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/360 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Epoch,Training Loss,Validation Loss
1,1.769329,1.740274
2,1.699381,1.713492


Training complete!


In [7]:
# Cell 7: Save adapter to Google Drive
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"Adapter saved to Google Drive ({total_size:.1f} MB)")

Adapter saved to Google Drive (17.2 MB)


In [8]:
# Cell 8: Quick test
test_input = pairs[-1]['input'][:2000]

prompt = (
    f"<s>[INST] You are an expert product analyst. "
    f"Given the following customer reviews, write a structured executive summary.\n\n"
    f"{test_input}\n[/INST]\n"
)

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = generated.split('[/INST]')[-1].strip()
print("=== Fine-tuned Output ===")
print(response)

=== Fine-tuned Output ===
## Product Issues Summary

**Overall Sentiment:** positive (8% negative reviews)

**Top Complaints:**
1. Fan Blade Pain — 1 mention — Hard plastic blades spin at high speed, causing pain when accidentally touched
2. No On/Off Switch — 1 mention — No convenient way to turn fan on/off, requiring manual touch of spinning blades
3. Limited Adjustability — 1 mention — Fan angle can't be easily adjusted without bending neck, potentially damaging USB port

**Key Insights:**
- Product is highly effective at cooling computer equipment, with multiple users reporting success in laptop cooling
- Despite spinning blades, users generally find the fan quiet during operation
- Durability is a concern with one user reporting a broken fan after 2 years of use
- The fan is considered a good value for its price point and functionality

**Recommendation:** Add a physical on/off switch and improve blade material/design to reduce accidental touch pain.


In [9]:
# Cell 9: Download adapter zip
!cd /content/drive/MyDrive && zip -r /content/mistral-qlora-adapter.zip mistral-qlora-adapter/
files.download('/content/mistral-qlora-adapter.zip')
print("Download complete!")

  adding: mistral-qlora-adapter/ (stored 0%)
  adding: mistral-qlora-adapter/README.md (deflated 65%)
  adding: mistral-qlora-adapter/adapter_model.safetensors (deflated 54%)
  adding: mistral-qlora-adapter/adapter_config.json (deflated 57%)
  adding: mistral-qlora-adapter/chat_template.jinja (deflated 64%)
  adding: mistral-qlora-adapter/tokenizer_config.json (deflated 48%)
  adding: mistral-qlora-adapter/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!


In [10]:
# Cell 10: Training history
history = trainer.state.log_history
for entry in history:
    if 'loss' in entry:
        print(f"Step {entry.get('step', '?')}: loss={entry['loss']:.4f}")
    if 'eval_loss' in entry:
        print(f"  eval_loss={entry['eval_loss']:.4f}")

Step 10: loss=2.5393
Step 20: loss=2.0834
Step 30: loss=1.8473
Step 40: loss=1.7693
  eval_loss=1.7403
Step 50: loss=1.7456
Step 60: loss=1.7169
Step 70: loss=1.7217
Step 80: loss=1.7088
Step 90: loss=1.6994
  eval_loss=1.7135
